# 实验二：FASTSim 整车能耗仿真

## 实验目标
- 理解整车能耗仿真的基本原理（动力学 + 效率 MAP）
- 用 FASTSim 加载真实车型，跑 UDDS 城市工况
- 对比不同车型的百公里电耗

## 背景
FASTSim 是 NREL 开发的整车仿真工具，内置 70 款真实车型参数。本实验跑一条标准城市驾驶循环，看电耗差异。

⚠️ 本 notebook 请用 **Python 3.10 (FASTSim)** 内核运行。

In [ ]:
import fastsim
print('FASTSim', fastsim.__version__)
from fastsim import cycle
from fastsim.simdrive import SimDrive
from fastsim.vehicle import Vehicle, DEFAULT_VEH_DB
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
print('all imports ok')

## 1. 加载循环工况
UDDS：美国城市驾驶循环，模拟市区走走停停。

In [ ]:
cyc = cycle.Cycle.from_file('udds')
t = np.array(cyc.time_s)
v = np.array(cyc.mps) * 3.6
print('时长: %.0f s, 步数: %d' % (t[-1], len(t)))

plt.figure(figsize=(10, 3))
plt.plot(t, v)
plt.xlabel('时间 (s)'); plt.ylabel('车速 (km/h)')
plt.title('UDDS 城市工况车速曲线')
plt.grid(True, alpha=0.3)
plt.show()

## 2. 选一款纯电动车，跑仿真

In [ ]:
db = pd.read_csv(DEFAULT_VEH_DB)
bevs = db[db['veh_pt_type'] == 'BEV']
print('纯电车型数:', len(bevs))
print(bevs[['veh_year', 'veh_pt_type']].head(3).to_string())

vnum = int(bevs.index[0])
veh = Vehicle.from_file(DEFAULT_VEH_DB, vnum=vnum)
print('\n车型 vnum=%d 加载成功' % vnum)

sd = SimDrive(cyc, veh)
sd.sim_drive()
print('仿真完成')

# 百公里电耗 (kWh/100km)
kwh_per_mi = float(np.array(sd.electric_kwh_per_mi).mean())
kwh_per_100km = kwh_per_mi * 62.137
print('百公里电耗: %.1f kWh/100km' % kwh_per_100km)

## 3. 对比两款车型

In [ ]:
results = {}
for idx in bevs.index[:2]:
    v = Vehicle.from_file(DEFAULT_VEH_DB, vnum=int(idx))
    s = SimDrive(cyc, v)
    s.sim_drive()
    e = float(np.array(s.electric_kwh_per_mi).mean()) * 62.137
    yr = db.loc[idx, 'veh_year']
    label = f'vnum={idx} ({yr})'
    results[label] = e
    print('%-20s %.1f kWh/100km' % (label, e))

plt.figure(figsize=(8, 4))
plt.bar(range(len(results)), list(results.values()))
plt.xticks(range(len(results)), list(results.keys()), rotation=10)
plt.ylabel('百公里电耗 (kWh/100km)')
plt.title('两款纯电动车 UDDS 工况能耗对比')
plt.tight_layout(); plt.show()

## 思考题
1. 为什么城市工况下电耗比高速工况低？（提示：动能回收）
2. 对比两款车的电耗差异，猜猜是什么参数导致的？（提示：看 veh_kg 车重）
3. 换成 HWFET 高速工况（`cycle.Cycle.from_file('hwfet')`），结果会反转吗？